# Predicción de Abandono de Curso

Proyecto de clasificación listo para **Google Colab**.

**Pregunta:** ¿podemos predecir qué estudiantes van a abandonar el curso?

**Target:** `abandono`

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset de ejemplo para predicción de abandono.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
url = "https://raw.githubusercontent.com/plotly/datasets/master/tips.csv"
df = pd.read_csv(url)
df["abandono"] = (df["tip"] < df["tip"].quantile(0.3)).astype(int)
print(df.shape)
print(df["abandono"].value_counts())
df.head()

## 2. EDA

In [ ]:
print("Tasa de abandono:", df["abandono"].mean().round(3))
df.describe().T

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="abandono")
plt.xticks([0, 1], ["Completó", "Abandonó"])
plt.title("Distribución de abandono")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="abandono", y="total_bill", ax=axes[0])
axes[0].set_title("Total vs Abandono")
sns.boxplot(data=df, x="abandono", y="size", ax=axes[1])
axes[1].set_title("Size vs Abandono")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="day", hue="abandono")
plt.title("Día vs Abandono")
plt.show()

### Hallazgos
- Variables numéricas correlacionan con abandono (simulado).
- Diferencias por día.
- Ideal para practicar predicción de abandono de curso.

## 3. Preparar datos

In [ ]:
X = df.drop(columns=["abandono"])
y = df["abandono"]

cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos

In [ ]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    resultados.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("roc_auc", ascending=False)

## 5. Evaluación

In [ ]:
mejor = "Random Forest"
pipe = pipes[mejor]
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["Completó", "Abandonó"]))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Completó", "Abandonó"]).plot(cmap="Blues")
plt.title("Matriz de confusión")
plt.show()

## 6. Feature importance

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 4))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Top features — Random Forest")
plt.tight_layout()
plt.show()

## 7. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_scaled = rf_pipe.named_steps["scaler"].transform(X_train)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_scaled)

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values[1], X_train_scaled, feature_names=X.columns, max_display=8, show=False)
plt.title("SHAP Summary — Abandono")
plt.tight_layout()
plt.show()

## 8. Conclusiones

- Variables numéricas influyen en el abandono (simulado).
- ROC-AUC suele ser alto en este ejemplo.
- Se puede extender a datasets reales de plataformas educativas.

**Rango esperado (ejemplo):** Accuracy ~0.80–0.90 | ROC-AUC ~0.85–0.95